# Ch02-01: 이산 마르코프 체인 — 모범답안


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
np.random.seed(42)

---
## 문제 1 풀이

고유값의 스펙트럼 갭으로 수렴 속도를 분석합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 다양한 전이 행렬
matrices = {
    '빠른 수렴': np.array([[0.5, 0.3, 0.2], [0.4, 0.3, 0.3], [0.3, 0.4, 0.3]]),
    '느린 수렴': np.array([[0.99, 0.005, 0.005], [0.005, 0.99, 0.005], [0.005, 0.005, 0.99]]),
    '주기적': np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0], [1.0, 0.0, 0.0]])
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, P) in zip(axes, matrices.items()):
    evals = np.linalg.eigvals(P)
    evals_sorted = sorted(np.abs(evals), reverse=True)
    spectral_gap = 1 - evals_sorted[1]
    mixing_time = int(np.ceil(1 / spectral_gap)) if spectral_gap > 0 else float('inf')

    # TV distance 수렴
    pi = np.ones(3) / 3  # 균등 초기
    dists = []
    stationary = np.real(np.linalg.eig(P.T)[1][:, np.argmin(np.abs(np.linalg.eigvals(P.T) - 1))])
    stationary = stationary / stationary.sum()
    current = np.array([1., 0., 0.])
    for step in range(100):
        tv = 0.5 * np.sum(np.abs(current - stationary))
        dists.append(tv)
        current = current @ P

    ax.plot(dists)
    ax.set_title(f'{name}\n|λ₂|={evals_sorted[1]:.4f}, gap={spectral_gap:.4f}')
    ax.set_xlabel('단계')
    ax.set_ylabel('총변동 거리')
    ax.set_yscale('log')
plt.tight_layout()
plt.show()

**결과 해석**: 스펙트럼 갭이 클수록 수렴이 빠릅니다. 주기적 체인은 λ₂의 절대값이 1이므로 수렴하지 않습니다(비주기 조건 위반).

---
## 문제 2 풀이

Power Iteration 기반 PageRank를 구현합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def pagerank(adj_matrix, d=0.85, tol=1e-8, max_iter=1000):
    n = adj_matrix.shape[0]
    out_degree = adj_matrix.sum(axis=1)
    out_degree[out_degree == 0] = 1  # dangling node 처리

    H = adj_matrix / out_degree[:, np.newaxis]
    G = d * H + (1 - d) / n * np.ones((n, n))

    pi = np.ones(n) / n
    for i in range(max_iter):
        pi_new = pi @ G
        if np.linalg.norm(pi_new - pi, 1) < tol:
            return pi_new, i + 1
        pi = pi_new
    return pi, max_iter

adj = np.array([
    [0,1,1,0,0,0],
    [0,0,1,1,0,0],
    [1,0,0,0,0,0],
    [0,0,0,0,1,1],
    [0,0,0,1,0,1],
    [0,0,0,1,0,0]
])

d_values = [0.5, 0.7, 0.85, 0.95]
fig, ax = plt.subplots(figsize=(10, 6))
for d in d_values:
    pr, iters = pagerank(adj, d=d)
    ax.bar(np.arange(6) + d_values.index(d)*0.2, pr, width=0.2, label=f'd={d}')
    print(f"d={d}: rank={np.argsort(-pr)+1}, iters={iters}")
ax.set_xticks(range(6))
ax.set_xticklabels([f'Page {i}' for i in range(6)])
ax.set_ylabel('PageRank')
ax.legend()
plt.title('감쇠 인자에 따른 PageRank')
plt.show()

**결과 해석**: d가 1에 가까울수록 링크 구조의 영향이 강해집니다. d=0.85가 표준값이며, dangling node 처리가 중요합니다.

---
## 문제 3 풀이

흡수 마르코프 체인으로 도박꾼의 파산 문제를 분석합니다.

In [ ]:
import numpy as np

def gamblers_ruin(N, p, initial_capital):
    # 상태: 0, 1, ..., N (0과 N은 흡수)
    # 일시 상태: 1, ..., N-1
    q = 1 - p
    n_transient = N - 1

    Q = np.zeros((n_transient, n_transient))
    for i in range(n_transient):
        state = i + 1
        if state - 1 >= 1: Q[i, i-1] = q  # 왼쪽
        if state + 1 <= N-1: Q[i, i+1] = p  # 오른쪽

    R = np.zeros((n_transient, 2))  # 흡수 상태: 0(파산), N(목표)
    for i in range(n_transient):
        state = i + 1
        if state - 1 == 0: R[i, 0] = q
        if state + 1 == N: R[i, 1] = p

    # 기본 행렬
    I = np.eye(n_transient)
    N_mat = np.linalg.inv(I - Q)

    # 흡수까지 기대 단계 수
    expected_steps = N_mat.sum(axis=1)

    # 흡수 확률
    B = N_mat @ R

    idx = initial_capital - 1
    print(f"N={N}, p={p}, 초기자본={initial_capital}")
    print(f"  파산 확률: {B[idx, 0]:.6f}")
    print(f"  목표 달성 확률: {B[idx, 1]:.6f}")
    print(f"  기대 게임 수: {expected_steps[idx]:.1f}")

    # 이론적 파산 확률 비교
    if p != 0.5:
        r = q / p
        theory = (r**initial_capital - r**N) / (1 - r**N)
    else:
        theory = 1 - initial_capital / N
    print(f"  이론적 파산 확률: {theory:.6f}")
    return B, expected_steps

for p in [0.4, 0.5, 0.6]:
    gamblers_ruin(N=10, p=p, initial_capital=5)
    print()

**결과 해석**: p<0.5(불리한 게임)이면 파산 확률이 기하급수적으로 1에 접근합니다. p=0.5이면 파산 확률은 1 - i/N이고, p>0.5에서도 파산이 가능합니다. 기본 행렬 N의 원소 n_ij는 상태 i에서 출발하여 흡수 전 상태 j를 방문하는 기대 횟수입니다.

---
## 확장 토론

마르코프 체인은 MCMC, 강화학습, 큐잉 이론의 기반입니다. 수렴 속도 분석은 MCMC 진단에 직접 사용됩니다.